In [ ]:
import pandas as pd

df = pd.read_csv("data/fed_speeches.csv")

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df['text'].iloc[0][:1000])

(1703, 4)
date     str
title    str
url      str
text     str
dtype: object
date     0
title    0
url      0
text     2
dtype: int64
It will not be easy reaching consensus on how to measure bank soundness and overall bank performance.  It cannot simply be done by observing market indicators.  For example, we cannot easily use the public ratings of holding company debt.  The ratings, after all, are achieved given the existence of the safety net .  The ratings are biased, therefore, from the perspective of achieving our stated goal--to impose prudential limits on banks as if there were no net.  In addition, I am sure that there would be disagreement between market participants and regulators over what should be acceptable debt ratings. The solution may be for the regulators to use the analytical tools developed by the market participants themselves for risk and performance assessment.  Regulators already have begun to move in this direction.  For example, beginning in January 1998, quali

In [9]:
# drop the 2 NaN rows
df = df.dropna(subset=["text"])
print(f"Speeches after dropping NaNs: {len(df)}")

# check text lengths
df["text_length"] = df["text"].str.len()
print(df["text_length"].describe())

# look at the shortest speeches
print(df.nsmallest(10, "text_length")[["date", "title", "text_length", "text"]])

Speeches after dropping NaNs: 1701
count     1701.000000
mean     20627.718989
std      10192.192263
min         37.000000
25%      13904.000000
50%      20398.000000
75%      26022.000000
max      75998.000000
Name: text_length, dtype: float64
            date                                              title  \
559   2005-12-14              Remarks on receipt of honorary degree   
569   2005-11-14  Stability and Economic Growth: The Role of the...   
1607   2/18/2022  Comments on "Some Benefits and Risks of a Hot ...   
417   2003-06-20         Central bank talk: Does it matter and why?   
2     1996-12-05  The challenge of central banking in a democrat...   
3     1996-12-03                 Clearinghouses and risk management   
9     1996-10-16            Technological advances and productivity   
4     1996-11-25  Supervisory and regulatory responses to financ...   
648    12/1/2006                                  Welcoming remarks   
10    1996-10-11                             

In [10]:
print(df["text"].iloc[0][-500:])

e do not operate in a vacuum--the activities of nonbank financial institutions are also important to the general well-being of our financial system and the macro economy. Regulators, of course, can only work with the framework laid down by Congress.  Let me conclude with the hope that this Congress will build on the experience of the last few years, including the experience with FDICIA, and take the next steps toward creating a structural and regulatory framework appropriate to the 21st century.


It appears as though a lot of speeches are suspiciously short, let's investigate this

In [11]:
# see how many speeches fall below various length thresholds
thresholds = [100, 500, 1000, 2000, 5000]
for t in thresholds:
    count = len(df[df["text_length"] < t])
    print(f"Under {t} chars: {count} speeches ({count/len(df)*100:.1f}%)")

# look at a few short ones to understand what went wrong
print(df[df["text_length"] < 500][["date", "title", "text_length", "text"]].head(10))

Under 100 chars: 2 speeches (0.1%)
Under 500 chars: 4 speeches (0.2%)
Under 1000 chars: 4 speeches (0.2%)
Under 2000 chars: 11 speeches (0.6%)
Under 5000 chars: 93 speeches (5.5%)
            date                                              title  \
417   2003-06-20         Central bank talk: Does it matter and why?   
559   2005-12-14              Remarks on receipt of honorary degree   
569   2005-11-14  Stability and Economic Growth: The Role of the...   
1607   2/18/2022  Comments on "Some Benefits and Risks of a Hot ...   

      text_length                                               text  
417           271  Paper by Governor Donald L. Kohn and Brian P. ...  
559            37            Return \r\n        to top 2005 Speeches  
569            37            Return \r\n        to top 2005 Speeches  
1607          261  February 18, 2022 Comments on "Some Benefits a...  


In [12]:
print(df[df["text_length"] > 10000]["text"].iloc[0][-500:])


on rate.  I think I have covered all the bases!  While I am focused on holding the line on inflation in the short run, I am also mindful of the importance of continuing progress toward price stability over the longer run and of the importance that current decisions be part of a longer strategic plan.  Nevertheless, you will not hear me complain or apologize if, over the next year, the chain GDP deflator remains near 2%, the unemployment rate remains near 5 1/2%, and the economy grows near trend.


In [13]:
# 1. drop NaNs
df = df.dropna(subset=["text"])

# 2. drop speeches that are too short to be useful
# 1000 chars is a reasonable cutoff — real speeches are much longer
df = df[df["text_length"] >= 1000]
print(f"Speeches after length filter: {len(df)}")

# 3. clean up whitespace and escape characters
df["text"] = df["text"].str.replace(r'\r\n', ' ', regex=True)
df["text"] = df["text"].str.replace(r'\s+', ' ', regex=True)
df["text"] = df["text"].str.strip()

# 4. parse dates properly
df["date"] = pd.to_datetime(df["date"], format="mixed")
df = df.sort_values("date").reset_index(drop=True)

# 5. check results
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Text length stats after cleaning:")
print(df["text_length"].describe())

# save cleaned version
df.to_csv("fed_speeches_cleaned.csv", index=False)
print("Saved to fed_speeches_cleaned.csv")

Speeches after length filter: 1697
Date range: 1996-06-13 00:00:00 to 2023-12-05 00:00:00
Text length stats after cleaning:
count     1697.000000
mean     20675.983500
std      10155.518981
min       1094.000000
25%      13924.000000
50%      20405.000000
75%      26032.000000
max      75998.000000
Name: text_length, dtype: float64
Saved to fed_speeches_cleaned.csv


In [14]:
# check speeches per year
print(df.groupby(df["date"].dt.year)["title"].count())

date
1996     19
1997     45
1998     57
1999     69
2000     62
2001     58
2002     76
2003     70
2004    102
2005     85
2006     73
2007     73
2008     73
2009     55
2010     60
2011     48
2012     42
2013     53
2014     41
2015     54
2016     44
2017     55
2018     44
2019     78
2020     53
2021     68
2022     45
2023     95
Name: title, dtype: int64


In [ ]:
# check what happened to 2001-2003 in the full unclean data
df_full = pd.read_csv("data/fed_speeches.csv")
df_full["date"] = pd.to_datetime(df_full["date"], format="mixed")

early = df_full[df_full["date"].dt.year.isin([2001, 2002, 2003])]
print(early[["date", "url", "text"]].head(20))
print(f"\nTotal 2001-2003 speeches: {len(early)}")
print(f"With text: {early['text'].notna().sum()}")
print(f"Text length distribution:")
print(early["text"].str.len().describe())

          date                                                url  \
252 2001-12-18  https://www.federalreserve.gov/boarddocs/speec...   
253 2001-12-05  https://www.federalreserve.gov/boarddocs/speec...   
254 2001-12-03  https://www.federalreserve.gov/boarddocs/speec...   
255 2001-11-30  https://www.federalreserve.gov/boarddocs/speec...   
256 2001-11-30  https://www.federalreserve.gov/boarddocs/speec...   
257 2001-11-27  https://www.federalreserve.gov/boarddocs/speec...   
258 2001-11-13  https://www.federalreserve.gov/boarddocs/speec...   
259 2001-11-08  https://www.federalreserve.gov/boarddocs/speec...   
260 2001-11-08  https://www.federalreserve.gov/boarddocs/speec...   
261 2001-10-26  https://www.federalreserve.gov/boarddocs/speec...   
262 2001-10-24  https://www.federalreserve.gov/boarddocs/speec...   
263 2001-10-23  https://www.federalreserve.gov/boarddocs/speec...   
264 2001-10-23  https://www.federalreserve.gov/boarddocs/speec...   
265 2001-10-16  https://www.federa

In [16]:
import requests
from bs4 import BeautifulSoup

url = early["url"].iloc[0]
print(url)

response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# try both selectors
article = soup.select_one("#article")
table = soup.find("table", {"width": "600"})

print("Article found:", article is not None)
print("Table found:", table is not None)

# print raw content area
content = soup.select_one("#content")
if content:
    print(content.prettify()[:2000])

https://www.federalreserve.gov/boarddocs/speeches/2001/20011218/default.htm
Article found: False
Table found: True


In [17]:
url = "https://www.federalreserve.gov/boarddocs/speeches/2001/20011218/default.htm"
response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# find ALL tables and print their content and width
for i, table in enumerate(soup.find_all("table")):
    width = table.get("width", "no width")
    text = table.get_text()[:100].strip()
    print(f"Table {i} | width={width} | {text}")

Table 0 | width=600 | Remarks by Governor Laurence H. Meyer
At the Center for Strategic and International Studies, Wa
Table 1 | width=600 | Financial Stability in Emerging Markets: What Have We Accomplished and What Remains To Be Done?
Table 2 | width=600 | Return to top
2001 Speeches


Ok, the mean and 25th percentile look good now, but there is still a minimum of 37 characters which means some short speeches are slipping through. Let's also make sure we parse the dates.

In [18]:
df_clean = df.copy()
df_clean["date"] = pd.to_datetime(df_clean["date"], format="mixed")
filt = df_clean["text_length"] >= 1000
df_clean = df_clean[filt]
print(f"speeches after length filter: {len(df_clean)}")

print(df_clean.groupby(df_clean["date"].dt.year)["title"].count())

speeches after length filter: 1697
date
1996     19
1997     45
1998     57
1999     69
2000     62
2001     58
2002     76
2003     70
2004    102
2005     85
2006     73
2007     73
2008     73
2009     55
2010     60
2011     48
2012     42
2013     53
2014     41
2015     54
2016     44
2017     55
2018     44
2019     78
2020     53
2021     68
2022     45
2023     95
Name: title, dtype: int64


Ok, so we have only 4 speeches less than 1000 characters and we have successfully recovered the early 2000s speeches. Let's finalize this dataset and get it moving into the model!

In [19]:
df_clean.to_csv("fed_speeches_cleaned.csv", index=False)

# Make sure everything looks right
print(f'Saved {len(df_clean)} cleaned speeches to fed_speeches_cleaned.csv')
print(f'Date Range: {df_clean["date"].min()} to {df_clean["date"].max()}')
print(f'Average Speech Length: {df_clean["text_length"].mean():.0f} chars')

Saved 1697 cleaned speeches to fed_speeches_cleaned.csv
Date Range: 1996-06-13 00:00:00 to 2023-12-05 00:00:00
Average Speech Length: 20676 chars
